In [ ]:
from pathlib import Path

import pandas as pd

from eutl_scraper import Settings
from eutl_scraper.eutl.extract.account_holders import load_accounts_power_bi_download
from eutl_scraper.eutl.extract.missing_accounts import (
    add_missing_accounts_from_transactions,
    get_transaction_parties,
)

e:\GIT\eutl_scraper_v2\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
settings = Settings("data_tmp/")
fn_trans = settings.fp("transactions", settings.dir_source)
fn_manual = Path("manual_data/accounts_20260412.xlsx")
fn_accounts = settings.fp("accounts", settings.dir_extracted)
fn_link_account_holder = settings.fp("link_account_holder", settings.dir_extracted)

## Load data

### Transaction parties

In [3]:
df_trans = pd.read_csv(fn_trans, low_memory=False)

In [4]:
df_transaction_parties = get_transaction_parties(df_trans)
df_transaction_parties.head(2).T

,0,1
registryName,Germany,United Kingdom
account_type1,121.0,121.0
account_type2,121-Person Holding Account,121-Person Holding Account
account_type3,0-None,0-None
openingDate,2005-05-20 00:00:00,2005-12-13 00:00:00
closingDate,NaN,2014-08-13 11:08:32
accountName,1914 - RWE Power AG Personenkonto,MLI Emissions Registry Account
account_identifier,1914,901
account_holder_name,RWE Power Aktiengesellschaft,Merrill Lynch International
account_holder_address1,RWE Power Aktiengesellschaft,Merrill Lynch Financial Centre


### Existing accounts

In [5]:
df_accounts = pd.read_csv(fn_accounts)
df_accounts.head(2).T

,0,1
accountName,"DONG Energy Generation A/S, SKV:DK317",Shell Trading International Limited (STIL):DK385
openingDate,2005-01-01,2005-02-08
closingDate,2012-11-16,2009-03-05
isClosurePending,False,False
snapshotDate,2026-04-19,2026-04-19
account_id,DK_317,DK_385
account_type,Former Operator Holding Account,Person Account in National Registry
created_at,2026-04-19 12:16:35.765387,2026-04-19 12:16:35.765387


### Manual accounts

In [6]:
df_manual_data = load_accounts_power_bi_download(fn_manual)

### Links account to holder

In [7]:
df_link = pd.read_csv(fn_link_account_holder)

## Extract data

### Get missing accounts

In [ ]:
df_account_full = add_missing_accounts_from_transactions(
    df_accounts, df_transaction_parties
)
df_account_full.info()

Missing accounts: 761
<class 'pandas.DataFrame'>
RangeIndex: 48777 entries, 0 to 48776
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   accountName       48777 non-null  str   
 1   openingDate       47981 non-null  str   
 2   closingDate       48016 non-null  str   
 3   isClosurePending  48016 non-null  object
 4   snapshotDate      48777 non-null  str   
 5   account_id        48777 non-null  str   
 6   account_type      47992 non-null  str   
 7   created_at        48777 non-null  str   
dtypes: object(1), str(7)
memory usage: 8.3+ MB


In [15]:
ids_linked_accounts = set(df_link.account_id.unique())
ids_accounts = set(df_accounts.account_id.unique())
ids_accounts_missing = ids_linked_accounts - ids_accounts
print(f"Number of accounts missing from accounts dataset: {len(ids_accounts_missing)}")

Number of accounts missing from accounts dataset: 0


In [14]:
list(ids_linked_accounts)[:10]

['GB_5008593',
 'GR_5046472',
 'LU_5042292',
 'DK_9494',
 'IT_3177',
 'DE_5014147',
 'DE_5056172',
 'PL_384',
 'AT_5004110',
 'PT_5053874']

In [13]:
list(ids_accounts)[:10]

['GB_5008593',
 'GR_5046472',
 'LU_5042292',
 'DK_9494',
 'IT_3177',
 'DE_5014147',
 'DE_5056172',
 'PL_384',
 'AT_5004110',
 'PT_5053874']